# 02 – Entrenamiento YOLOv11

**Proyecto:** Detección Automática de Fracturas Óseas en Radiografías  
**Materia:** Visión por Computadora II – CEIA/FIUBA  
**Autores:** Lucia T. Capon Paul · Cesar Orellana · Leandro Britez

---

## Objetivos
- Fine-tuning de YOLOv11 sobre el dataset de fracturas óseas.
- Explorar el impacto de distintas técnicas de Data Augmentation.
- Registrar métricas por época para análisis posterior.

In [ ]:
import sys
sys.path.insert(0, '..')

from ultralytics import YOLO
import yaml
from pathlib import Path
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Descarga del dataset desde Roboflow

In [ ]:
# Descomentar y completar con tu API key de Roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("veda").project("bone-fracture-detection-daoon")
# version = project.version(1)
# dataset = version.download("yolov11", location="../data/bone-fracture-detection-daoon-1")

DATA_YAML = Path('../data/bone-fracture-detection-daoon-1/data.yaml')
assert DATA_YAML.exists(), 'dataset no encontrado, seguí las instrucciones en data/README.md'
print(f'Dataset YAML: {DATA_YAML}')

## 2. Configuración del experimento

In [ ]:
with open('../configs/yolov11.yaml') as f:
    cfg = yaml.safe_load(f)

print('Configuración cargada:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

## 3. Carga del modelo preentrenado

In [ ]:
model = YOLO(cfg['model_weights'])
print(f'Modelo cargado: {cfg["model_weights"]}')
print(model.info())

## 4. Fine-tuning

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=cfg['epochs'],
    imgsz=cfg['imgsz'],
    batch=cfg['batch'],
    lr0=cfg['lr0'],
    lrf=cfg['lrf'],
    momentum=cfg['momentum'],
    weight_decay=cfg['weight_decay'],
    # Data augmentation
    hsv_h=cfg['hsv_h'],
    hsv_s=cfg['hsv_s'],
    hsv_v=cfg['hsv_v'],
    flipud=cfg['flipud'],
    fliplr=cfg['fliplr'],
    mosaic=cfg['mosaic'],
    mixup=cfg['mixup'],
    # Output
    project='../results',
    name='yolov11_fracture',
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)

print('Entrenamiento finalizado.')
print(f'Mejor checkpoint: {results.save_dir}/weights/best.pt')

## 5. Evaluación rápida en validación

In [ ]:
val_results = model.val(data=str(DATA_YAML), split='val')
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:    {val_results.box.mp:.4f}')
print(f'Recall:       {val_results.box.mr:.4f}')

---
## Notas del experimento

*(Completar tras el entrenamiento)*

- **Variante de modelo usada:** ...
- **Épocas entrenadas / early stopping:** ...
- **mAP@0.5 final:** ...
- **Observaciones:** ...